In [2]:
import numpy as np

from rowland_geometry.lens import (
    meridional_p,
    meridional_q,
    sagittal_p,
    sagittal_q,
)

# ------------------------------------------------------------
# Test geometries
# ------------------------------------------------------------

cases = [
    (500.0, 70.0, 0.0),
    (500.0, 70.0, 10.0),
    (500.0, 70.0, -10.0),
    (1000.0, 60.0, 5.0),
    (300.0, 80.0, 12.0),
]

print("ROWLAND CLOSED-FORM / LENS-EQUATION CONSISTENCY CHECKS\n")

for R, bragg, asymmetry in cases:

    theta1 = bragg + asymmetry
    theta2 = bragg - asymmetry

    t1 = np.radians(theta1)
    t2 = np.radians(theta2)

    # --------------------------------------------------------
    # On-circle source distance
    # --------------------------------------------------------

    rho = R * np.sin(t1)

    # --------------------------------------------------------
    # Closed-form Rowland expressions
    # --------------------------------------------------------

    fm_expected = R * np.sin(t2)

    sin_ba = np.sin(np.radians(bragg + asymmetry))
    sin_b = np.sin(np.radians(bragg))
    cos_a = np.cos(np.radians(asymmetry))

    fs_expected = (
        R * sin_ba
        / (
            2.0 * sin_b * cos_a * sin_ba
            - 1.0
        )
    )

    # --------------------------------------------------------
    # General lens equations
    # --------------------------------------------------------

    fm_lens = meridional_q(
        p=rho,
        theta1=theta1,
        theta2=theta2,
        radius=R,
    )

    fs_lens = sagittal_q(
        p=rho,
        theta1=theta1,
        theta2=theta2,
        radius=R,
    )

    # --------------------------------------------------------
    # Inverse checks
    # --------------------------------------------------------

    rho_from_fm = meridional_p(
        q=fm_lens,
        theta1=theta1,
        theta2=theta2,
        radius=R,
    )

    rho_from_fs = sagittal_p(
        q=fs_lens,
        theta1=theta1,
        theta2=theta2,
        radius=R,
    )

    # --------------------------------------------------------
    # Comparisons
    # --------------------------------------------------------

    fm_ok = np.isclose(fm_lens, fm_expected)
    fs_ok = np.isclose(fs_lens, fs_expected)

    meridional_inverse_ok = np.isclose(rho_from_fm, rho)
    sagittal_inverse_ok = np.isclose(rho_from_fs, rho)

    print(
        f"R = {R:7.1f} mm, "
        f"Bragg = {bragg:6.2f}°, "
        f"alpha = {asymmetry:+6.2f}°"
    )

    print(
        f"  rho:          {rho:12.6f} mm"
    )

    print(
        f"  Meridional:   "
        f"lens = {fm_lens:12.6f}, "
        f"closed form = {fm_expected:12.6f}, "
        f"{'PASS' if fm_ok else 'FAIL'}"
    )

    print(
        f"  Sagittal:     "
        f"lens = {fs_lens:12.6f}, "
        f"closed form = {fs_expected:12.6f}, "
        f"{'PASS' if fs_ok else 'FAIL'}"
    )

    print(
        f"  Inverse p from meridional q: "
        f"{'PASS' if meridional_inverse_ok else 'FAIL'}"
    )

    print(
        f"  Inverse p from sagittal q:   "
        f"{'PASS' if sagittal_inverse_ok else 'FAIL'}"
    )

    print()

# ------------------------------------------------------------
# Final assertions
# ------------------------------------------------------------

for R, bragg, asymmetry in cases:

    theta1 = bragg + asymmetry
    theta2 = bragg - asymmetry

    rho = R * np.sin(np.radians(theta1))

    fm_expected = R * np.sin(np.radians(theta2))

    sin_ba = np.sin(np.radians(bragg + asymmetry))
    sin_b = np.sin(np.radians(bragg))
    cos_a = np.cos(np.radians(asymmetry))

    fs_expected = (
        R * sin_ba
        / (
            2.0 * sin_b * cos_a * sin_ba
            - 1.0
        )
    )

    fm_lens = meridional_q(
        rho,
        theta1,
        theta2,
        R,
    )

    fs_lens = sagittal_q(
        rho,
        theta1,
        theta2,
        R,
    )

    assert np.isclose(fm_lens, fm_expected)
    assert np.isclose(fs_lens, fs_expected)

    assert np.isclose(
        meridional_p(
            fm_lens,
            theta1,
            theta2,
            R,
        ),
        rho,
    )

    assert np.isclose(
        sagittal_p(
            fs_lens,
            theta1,
            theta2,
            R,
        ),
        rho,
    )

print("All closed-form / lens-equation checks passed.")

ROWLAND CLOSED-FORM / LENS-EQUATION CONSISTENCY CHECKS

R =   500.0 mm, Bragg =  70.00°, alpha =  +0.00°
  rho:            469.846310 mm
  Meridional:   lens =   469.846310, closed form =   469.846310, PASS
  Sagittal:     lens =   613.340798, closed form =   613.340798, PASS
  Inverse p from meridional q: PASS
  Inverse p from sagittal q:   PASS

R =   500.0 mm, Bragg =  70.00°, alpha = +10.00°
  rho:            492.403877 mm
  Meridional:   lens =   433.012702, closed form =   433.012702, PASS
  Sagittal:     lens =   598.510992, closed form =   598.510992, PASS
  Inverse p from meridional q: PASS
  Inverse p from sagittal q:   PASS

R =   500.0 mm, Bragg =  70.00°, alpha = -10.00°
  rho:            433.012702 mm
  Meridional:   lens =   492.403877, closed form =   492.403877, PASS
  Sagittal:     lens =   718.253946, closed form =   718.253946, PASS
  Inverse p from meridional q: PASS
  Inverse p from sagittal q:   PASS

R =  1000.0 mm, Bragg =  60.00°, alpha =  +5.00°
  rho:       

In [3]:
import numpy as np

from rowland_geometry.rowland_circle import RowlandCircle
from rowland_geometry.lens import (
    meridional_q,
    sagittal_q,
)

# ------------------------------------------------------------
# Test geometries
# ------------------------------------------------------------

cases = [
    (500.0, 70.0, 0.0),
    (500.0, 70.0, 10.0),
    (500.0, 70.0, -10.0),
    (1000.0, 60.0, 5.0),
    (300.0, 80.0, 12.0),
]

print("ROWLANDCIRCLE / LENS.PY CONSISTENCY CHECKS\n")

for diameter, bragg, asymmetry in cases:

    circle = RowlandCircle(
        diameter=diameter,
        bragg=bragg,
        asymmetry=asymmetry,
    )

    theta1 = bragg + asymmetry
    theta2 = bragg - asymmetry

    rho = circle.chord_rho

    fm_lens = meridional_q(
        p=rho,
        theta1=theta1,
        theta2=theta2,
        radius=diameter,
    )

    fs_lens = sagittal_q(
        p=rho,
        theta1=theta1,
        theta2=theta2,
        radius=diameter,
    )

    fm_ok = np.isclose(
        circle.chord_fm,
        fm_lens,
    )

    fs_ok = np.isclose(
        circle.chord_fs,
        fs_lens,
    )

    print(
        f"R = {diameter:7.1f} mm, "
        f"Bragg = {bragg:6.2f}°, "
        f"alpha = {asymmetry:+6.2f}°"
    )

    print(
        f"  rho:        "
        f"RowlandCircle = {circle.chord_rho:12.6f}"
    )

    print(
        f"  fm:         "
        f"RowlandCircle = {circle.chord_fm:12.6f}, "
        f"lens.py = {fm_lens:12.6f}, "
        f"{'PASS' if fm_ok else 'FAIL'}"
    )

    print(
        f"  fs:         "
        f"RowlandCircle = {circle.chord_fs:12.6f}, "
        f"lens.py = {fs_lens:12.6f}, "
        f"{'PASS' if fs_ok else 'FAIL'}"
    )

    print()

# ------------------------------------------------------------
# Final assertions
# ------------------------------------------------------------

for diameter, bragg, asymmetry in cases:

    circle = RowlandCircle(
        diameter=diameter,
        bragg=bragg,
        asymmetry=asymmetry,
    )

    theta1 = bragg + asymmetry
    theta2 = bragg - asymmetry

    fm_lens = meridional_q(
        p=circle.chord_rho,
        theta1=theta1,
        theta2=theta2,
        radius=diameter,
    )

    fs_lens = sagittal_q(
        p=circle.chord_rho,
        theta1=theta1,
        theta2=theta2,
        radius=diameter,
    )

    assert np.isclose(
        circle.chord_fm,
        fm_lens,
    )

    assert np.isclose(
        circle.chord_fs,
        fs_lens,
    )

print("All RowlandCircle / lens.py checks passed.")

ROWLANDCIRCLE / LENS.PY CONSISTENCY CHECKS

R =   500.0 mm, Bragg =  70.00°, alpha =  +0.00°
  rho:        RowlandCircle =   469.846310
  fm:         RowlandCircle =   469.846310, lens.py =   469.846310, PASS
  fs:         RowlandCircle =   613.340798, lens.py =   613.340798, PASS

R =   500.0 mm, Bragg =  70.00°, alpha = +10.00°
  rho:        RowlandCircle =   492.403877
  fm:         RowlandCircle =   433.012702, lens.py =   433.012702, PASS
  fs:         RowlandCircle =   598.510992, lens.py =   598.510992, PASS

R =   500.0 mm, Bragg =  70.00°, alpha = -10.00°
  rho:        RowlandCircle =   433.012702
  fm:         RowlandCircle =   492.403877, lens.py =   492.403877, PASS
  fs:         RowlandCircle =   718.253946, lens.py =   718.253946, PASS

R =  1000.0 mm, Bragg =  60.00°, alpha =  +5.00°
  rho:        RowlandCircle =   906.307787
  fm:         RowlandCircle =   819.152044, lens.py =   819.152044, PASS
  fs:         RowlandCircle =  1607.505346, lens.py =  1607.505346, PASS

